In [0]:
CREATE TABLE ecommerce_mvp.gold.dim_cliente AS SELECT * FROM ecommerce_mvp.silver.clientes;
CREATE TABLE ecommerce_mvp.gold.dim_produto AS SELECT * FROM ecommerce_mvp.silver.produtos;

CREATE TABLE ecommerce_mvp.gold.fato_vendas AS
SELECT ip.id_item, ip.id_pedido, ip.id_cliente, ip.id_produto,
       p.categoria, p.marca, p.departamento,
       ip.preco_venda, p.custo, (ip.preco_venda - p.custo) AS margem,
       ip.data_criacao AS data_venda, ped.status AS status_pedido
FROM ecommerce_mvp.silver.itens_pedido ip
JOIN ecommerce_mvp.silver.produtos p ON ip.id_produto = p.id_produto
JOIN ecommerce_mvp.silver.pedidos ped ON ip.id_pedido = ped.id_pedido
WHERE ped.status NOT IN ('Cancelled', 'Returned');

CREATE TABLE ecommerce_mvp.gold.estoque_atual AS
SELECT id_produto, count(*) AS unidades_em_estoque,
       AVG(DATEDIFF(current_date(), data_entrada_estoque)) AS dias_medios_parado
FROM ecommerce_mvp.silver.estoque
WHERE em_estoque_atualmente = true
GROUP BY id_produto;

CREATE TABLE ecommerce_mvp.gold.giro_vendas_produto AS
SELECT id_produto, categoria, COUNT(*) AS unidades_vendidas_total,
       COUNT(CASE WHEN data_venda >= date_sub(current_date(), 90) THEN 1 END) AS unidades_vendidas_90d,
       SUM(preco_venda) AS receita_total
FROM ecommerce_mvp.gold.fato_vendas
GROUP BY id_produto, categoria;

CREATE TABLE ecommerce_mvp.gold.kpis_mensais AS
SELECT DATE_TRUNC('month', data_venda) AS mes, SUM(preco_venda) AS receita_total,
       COUNT(DISTINCT id_pedido) AS qtd_pedidos,
       SUM(preco_venda) / COUNT(DISTINCT id_pedido) AS ticket_medio,
       COUNT(DISTINCT id_cliente) AS clientes_ativos,
       COUNT(DISTINCT id_pedido) / COUNT(DISTINCT id_cliente) AS frequencia_media
FROM ecommerce_mvp.gold.fato_vendas
GROUP BY DATE_TRUNC('month', data_venda);

CREATE TABLE ecommerce_mvp.gold.ciclo_vida_cliente AS
WITH compras_mensais AS (
    SELECT id_cliente, DATE_TRUNC('month', data_venda) AS mes
    FROM ecommerce_mvp.gold.fato_vendas
    GROUP BY id_cliente, DATE_TRUNC('month', data_venda)
),
com_mes_anterior AS (
    SELECT *, LAG(mes) OVER (PARTITION BY id_cliente ORDER BY mes) AS mes_anterior
    FROM compras_mensais
)
SELECT mes, id_cliente,
    CASE WHEN mes_anterior IS NULL THEN 'Novo'
         WHEN MONTHS_BETWEEN(mes, mes_anterior) <= 2 THEN 'Retido'
         ELSE 'Reativado' END AS classificacao
FROM com_mes_anterior;

CREATE TABLE ecommerce_mvp.gold.base_crm AS
WITH ultima_compra AS (
    SELECT id_cliente, MAX(data_venda) AS data_ultima_compra,
           COUNT(DISTINCT id_pedido) AS total_pedidos, SUM(preco_venda) AS receita_total_cliente
    FROM ecommerce_mvp.gold.fato_vendas
    GROUP BY id_cliente
)
SELECT id_cliente, data_ultima_compra, total_pedidos, receita_total_cliente,
    DATEDIFF(current_date(), data_ultima_compra) AS dias_desde_ultima_compra,
    CASE WHEN DATEDIFF(current_date(), data_ultima_compra) > 180 AND total_pedidos > 1 THEN 'Alvo de reativação'
         WHEN total_pedidos >= 3 THEN 'Cliente recorrente de valor'
         ELSE 'Acompanhamento padrão' END AS segmento_crm
FROM ultima_compra;



In [0]:
CREATE TABLE ecommerce_mvp.gold.funil_conversao AS
SELECT
    id_sessao,
    MAX(id_cliente) AS id_cliente,
    MAX(origem_trafego) AS origem_trafego,
    MAX(CASE WHEN tipo_evento IN ('home', 'department') THEN 1 ELSE 0 END) AS viu_home,
    MAX(CASE WHEN tipo_evento = 'product' THEN 1 ELSE 0 END) AS viu_produto,
    MAX(CASE WHEN tipo_evento = 'cart' THEN 1 ELSE 0 END) AS adicionou_carrinho,
    MAX(CASE WHEN tipo_evento = 'purchase' THEN 1 ELSE 0 END) AS comprou
FROM ecommerce_mvp.silver.eventos
GROUP BY id_sessao;